# Nonconvex Forward Example

Define nonconvex data, training, and projection settings in the notebook, then run the same builder-owned runner used by `main.py`.


## Setup
Import the builder runner and create helpers for notebook-local run folders and summaries.


In [1]:
from argparse import Namespace
from pathlib import Path
import json

from kkthn.builder import ProblemBuilder


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def make_output_root(name):
    output_root = ROOT / "notebooks" / "_runs" / name
    output_root.mkdir(parents=True, exist_ok=True)
    return output_root


def latest_run_dir(output_root):
    runs = [path for path in output_root.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found under {output_root}")
    return max(runs, key=lambda path: path.stat().st_mtime)


def load_summary(run_dir):
    with open(run_dir / "summary.json", "r", encoding="utf-8") as fh:
        return json.load(fh)


def standard_args(problem_type, data, config, output_root):
    return Namespace(
        type=problem_type,
        action="run",
        mode="forward",
        p=data.get("p"),
        n=data.get("n"),
        me=data.get("me"),
        mi=data.get("mi"),
        samples=data.get("num_samples", data.get("N_points", data.get("N_samples"))),
        epochs=config.get("epochs"),
        batch_size=config.get("batch_size"),
        learning_rate=config.get("learning_rate"),
        solver=data.get("solver"),
        train_frac=config.get("train_frac"),
        hidden_size=config.get("hidden_size"),
        hidden_layers=config.get("hidden_layers"),
        seed=data.get("seed"),
        noise_scale=data.get("noise_scale"),
        output_dir=str(output_root),
    )


## Configuration
Edit these dictionaries to change the generated problem, training loop, or projection settings.


In [2]:
DATA = {
    "type": "nonconvex",
    "p": 1,
    "n": 3,
    "me": 1,
    "mi": 1,
    "num_samples": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "is_diag_Q": False,
    "force_regenerate": True,
}

CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


## Train
Create CLI-like arguments and call `ProblemBuilder.run(...)` with the notebook dictionaries.


In [3]:
output_root = make_output_root("nonconvex_forward")
args = standard_args("nonconvex", DATA, CONFIG, output_root)
exit_code = ProblemBuilder.run(args, root=ROOT, data=DATA, train=CONFIG, proj=PROJ)
run_dir = latest_run_dir(output_root)
summary_json = load_summary(run_dir)

print(f"Exit code: {exit_code}")
print(f"Run directory: {run_dir}")
print(f"Saved artifacts: {summary_json['artifacts']}")


KKT-HardNet runner | type=nonconvex action=run mode=forward
Case directory: D:\Projects\KKTHardNet\case\nonconvx
Data config: {'type': 'nonconvex', 'p': 1, 'n': 3, 'me': 1, 'mi': 1, 'num_samples': 12, 'seed': 42, 'noise_scale': 0.0, 'is_diag_Q': False, 'force_regenerate': True}
Train config: {'epochs': 3, 'batch_size': 4, 'learning_rate': 0.001, 'train_frac': 0.8, 'hidden_size': 32, 'hidden_layers': 2, 'seed': 42, 'dtype': 'float64', 'print_every': 1, 'hidden_dim': 2}
Projection config: {'fb_eps': 1e-08, 'gn_max_iters': 25, 'gn_tol': 1e-08}
Labels: X=(12, 1) Y=(12, 3) source=optimizer
KKT-HardNet
  dims: n_x=1 n_y=3 n_eq=1 n_ineq=7
  mode: forward
  samples: train=9 val=3 batch_size=4
  network: [1, 32, 32, 3]
epoch 0001 | train=7.232340e-01 val=7.020113e-01 raw_val=1.122610e+00 eq=2.837e-13 ineq=1.768e-12 | train_batch=0.8254s val_batch=0.0012s
epoch 0002 | train=6.697706e-01 val=6.471397e-01 raw_val=1.046630e+00 eq=2.043e-12 ineq=1.082e-12 | train_batch=0.0011s val_batch=0.0014s
epoc

## Summary
Read the emitted `summary.json` from the newest run folder.


In [4]:
summary = {
    "output_dir": str(run_dir),
    "dims": summary_json["dims"],
    "final": summary_json["final"],
    "component_time_percent": summary_json["timing_profile"]["component_time_percent"],
}
print(json.dumps(summary, indent=2))


{
  "output_dir": "D:\\Projects\\KKTHardNet\\notebooks\\_runs\\nonconvex_forward\\nonconvx_20260414_121701",
  "dims": {
    "n_eq": 1,
    "n_ineq": 7,
    "n_x": 1,
    "n_y": 3,
    "n_z": 18,
    "raw_n_ineq": 1
  },
  "final": {
    "epoch": 3,
    "mean_batch_loss": 0.6499664441009584,
    "train_batches": 3,
    "train_epoch_time_sec": 0.00346190000000135,
    "train_eq_l2": 1.0426752054352542e-12,
    "train_eval_time_sec": 0.000992300000000057,
    "train_ineq_l2": 4.0370176337647636e-13,
    "train_loss": 0.6225169037553852,
    "train_raw_mse": 0.8404443627219955,
    "train_step_time_per_batch_sec": 0.000652500000001055,
    "train_step_time_sec": 0.001957500000003165,
    "train_time_per_batch_sec": 0.0011539666666671167,
    "val_eq_l2": 1.926042658695337e-12,
    "val_ineq_l2": 6.032026729959246e-13,
    "val_loss": 0.5954274222692348,
    "val_raw_mse": 0.9780708925948778,
    "validation_batches": 1,
    "validation_epoch_time_sec": 0.0010907000000024425,
    "validati